# LeetCode 3487 — Maximum Unique Subarray Sum After Deletion bằng Python

Link đề: https://leetcode.com/problems/maximum-unique-subarray-sum-after-deletion/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, Greedy

Skeleton LeetCode:

```python
class Solution:
    def maxSum(self, nums: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: được **xoá tuỳ ý** các phần tử (nhưng **không** được xoá hết), sau đó chọn một **mảng con liên tiếp** có các phần tử **đôi một khác nhau** sao cho tổng **lớn nhất**.
2. Nhận ra rằng chính việc được **xoá** biến bài toán "cửa sổ trượt" thành bài toán **tham lam**: gom mọi giá trị dương **khác nhau** lại với nhau.
3. Xử lý nhánh đặc biệt: khi **không có số dương** (toàn `<= 0`) thì đáp án là **phần tử lớn nhất** (vì buộc phải giữ lại ít nhất một số).
4. Viết 4 cách: brute force theo định nghĩa (liệt kê mọi cách xoá), tập hợp `set`, sắp xếp giảm dần, và bảng đếm theo giá trị; đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: cộng cả số trùng, trả `0` khi toàn số âm, cộng cả số âm, và quên rằng được xoá (dùng cửa sổ trượt trên mảng gốc).

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Bạn được **xoá bất kỳ số lượng phần tử nào** khỏi `nums` nhưng **không được làm mảng rỗng**. Sau khi xoá, hãy chọn một **mảng con** (subarray, các phần tử **liên tiếp** của mảng sau khi xoá) sao cho:

- tất cả phần tử trong mảng con **đôi một khác nhau** (unique), và
- **tổng** các phần tử của nó **lớn nhất** có thể.

Trả về tổng lớn nhất đó.

| Ví dụ | nums | Output | Giải thích |
|---|---|---|---|
| 1 | `[1,2,3,4,5]` | 15 | không cần xoá; cả mảng đều khác nhau, tổng 15 |
| 2 | `[1,1,0,1,1]` | 1 | xoá bớt, còn `[1]` (hoặc `[1,0]`): tổng lớn nhất là 1 |
| 3 | `[1,2,-1,-2,1,0,-1]` | 3 | xoá các số âm và bản `1` thứ hai, còn `[1,2,0]`: tổng 3 |

Constraints: `1 <= nums.length <= 100`; `-100 <= nums[i] <= 100`.

In [1]:
nums = [1, 2, -1, -2, 1, 0, -1]
positives = set()
for x in nums:
    if x > 0:
        positives.add(x)
print("các giá trị dương khác nhau:", sorted(positives))
print("tổng =", sum(positives))

các giá trị dương khác nhau: [1, 2]
tổng = 3


## 3. Kiến thức nền

### 3.1. Mảng con (subarray) và mảng con "unique"

**Mảng con** là một đoạn **liên tiếp** của mảng. Mảng con **unique** là mảng con mà **không có hai phần tử bằng nhau**. Ví dụ `[2, 1, 0]` unique; `[1, 2, 1]` không unique (hai số `1`). Điểm quan trọng của đề: mảng con được chọn **sau khi xoá**, nên các phần tử vốn cách xa nhau có thể trở thành liền kề.

In [2]:
def is_unique(a):
    return len(set(a)) == len(a)

print(is_unique([2, 1, 0]), is_unique([1, 2, 1]), is_unique([5]))
nums = [1, 2, -1, -2, 1, 0, -1]
# xoá các phần tử ở chỉ số 2, 3, 4 (giữ 1, 2, 0, -1)
after = [v for i, v in enumerate(nums) if i not in (2, 3, 4)]
print("sau khi xoá:", after, "| unique?", is_unique(after))

True False True
sau khi xoá: [1, 2, 0, -1] | unique? True


### 3.2. `set` — tập hợp loại bỏ trùng

`set` lưu mỗi giá trị **một lần**, thêm/kiểm tra phần tử trung bình `O(1)`. `set(nums)` cho các giá trị **khác nhau** của `nums`. Có thể lọc bằng biểu thức: `{x for x in nums if x > 0}` là tập các số dương khác nhau (set comprehension).

In [3]:
nums = [1, 1, 0, 1, 1]
print(set(nums), len(set(nums)))
print({x for x in [1, 2, -1, -2, 1, 0, -1] if x > 0})

{0, 1} 2
{1, 2}


### 3.3. `max`, `sum` và `any`

`max(nums)` là phần tử lớn nhất; `sum(iterable)` là tổng; `any(cond for ...)` kiểm tra có phần tử nào thoả. Nhánh "không có số dương" có thể viết `if max(nums) <= 0`.

In [4]:
nums = [-3, -1, -2]
print(max(nums), sum({x for x in nums if x > 0}), max(nums) <= 0)

-1 0 True


### 3.4. `itertools.combinations` / mặt nạ bit để liệt kê "cách xoá"

Mỗi cách xoá tương ứng một **tập con khác rỗng** các chỉ số được giữ lại. Với `n` phần tử có `2^n - 1` tập con khác rỗng: ta duyệt `mask` từ `1` tới `2^n - 1`, bit `i` bằng `1` nghĩa là giữ `nums[i]`. Cách này chậm (`2^n`) nhưng **trung thực với định nghĩa đề** nên rất tốt để kiểm chứng các cách nhanh trên mảng nhỏ.

In [5]:
nums = [4, 5, 6]
for mask in range(1, 2 ** len(nums)):
    kept = [nums[i] for i in range(len(nums)) if mask >> i & 1]
    print(f"mask={mask:03b} -> giữ {kept}")

mask=001 -> giữ [4]
mask=010 -> giữ [5]
mask=011 -> giữ [4, 5]
mask=100 -> giữ [6]
mask=101 -> giữ [4, 6]
mask=110 -> giữ [5, 6]
mask=111 -> giữ [4, 5, 6]


### 3.5. Mảng đếm theo giá trị (bucket)

Vì `-100 <= nums[i] <= 100`, ta có thể dùng mảng `seen` độ dài `201`, ô `seen[x + 100]` ghi "giá trị `x` đã xuất hiện". Đây là một `set` tự làm, truy cập `O(1)` và không dùng hash.

In [6]:
seen = [False] * 201
for x in [1, 2, -1, -2, 1, 0, -1]:
    seen[x + 100] = True
print([v - 100 for v in range(201) if seen[v]])

[-2, -1, 0, 1, 2]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Xét `nums = [1, 2, -1, -2, 1, 0, -1]`. Vì được **xoá tuỳ ý**, ta có thể "ép" mọi phần tử mình thích thành **liền kề**: giữ các số ta muốn, xoá hết còn lại. Muốn tổng lớn nhất:

- **Số dương** khác nhau: giữ **một** bản của mỗi giá trị (`1` và `2`) -> `+3`.
- **Số `0`**: cộng thêm `0`, vô ích nhưng không hại.
- **Số âm**: giữ chỉ làm tổng giảm -> xoá hết.

Vậy đáp án = **tổng các giá trị dương khác nhau** = `1 + 2 = 3`.

Ngoại lệ: nếu **không có số dương** (ví dụ `[-3, -1, -2]`), ta vẫn **phải** giữ lại ít nhất một phần tử; chọn phần tử **lớn nhất** (gần `0` nhất): `-1`. Với `[0, -5]` đáp án là `0`.

## 5. Cách 1 — Brute Force / baseline

Làm **đúng định nghĩa** đề: liệt kê mọi tập chỉ số giữ lại (khác rỗng) bằng mặt nạ bit; với mỗi mảng sau xoá, thử mọi mảng con `[i..j]`, kiểm tra unique (bằng `set`) và lấy tổng lớn nhất.

Nút thắt: `2^n` cách xoá × `O(n^2)` mảng con × kiểm tra -> bùng nổ; chỉ chạy được với `n` cỡ 12. Ta dùng nó làm **chuẩn chân lý** cho mảng nhỏ, không dùng để nộp bài.

In [7]:
class SolutionBrute:
    def maxSum(self, nums):
        n = len(nums)
        best = None
        for mask in range(1, 2 ** n):
            arr = [nums[i] for i in range(n) if mask >> i & 1]
            m = len(arr)
            for i in range(m):
                seen = set()
                total = 0
                for j in range(i, m):
                    if arr[j] in seen:
                        break
                    seen.add(arr[j])
                    total += arr[j]
                    if best is None or total > best:
                        best = total
        return best

print(SolutionBrute().maxSum([1, 2, 3, 4, 5]))
print(SolutionBrute().maxSum([1, 1, 0, 1, 1]))
print(SolutionBrute().maxSum([1, 2, -1, -2, 1, 0, -1]))

15
1
3


**Độ phức tạp:** Time `O(2^n * n^2)` (mảng con đến khi gặp trùng đầu tiên), Space `O(n)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** vì được xoá tuỳ ý, **bất kỳ tập giá trị đôi một khác nhau nào** đều có thể trở thành một mảng con liên tiếp (giữ một bản mỗi giá trị, xoá phần còn lại). Do đó bài trở thành: **chọn một tập con các giá trị khác nhau, khác rỗng, tổng lớn nhất**.

**Quy tắc "nếu… thì…":**

- Nếu `max(nums) <= 0` -> không có số dương: đáp án là `max(nums)` (buộc giữ ít nhất một số; chọn số lớn nhất).
- Ngược lại -> đáp án là **tổng các giá trị dương khác nhau**. Số `0` thêm vào không đổi tổng; số âm thì xoá.

Invariant: tập đã chọn luôn gồm các giá trị **dương** và **đôi một khác nhau**; thêm bất kỳ giá trị dương mới nào chỉ làm tổng tăng, thêm giá trị `<= 0` không làm tổng tăng.

In [8]:
def greedy(nums):
    m = max(nums)
    if m <= 0:
        return m
    return sum({x for x in nums if x > 0})

for nums in ([1, 2, 3, 4, 5], [1, 1, 0, 1, 1], [1, 2, -1, -2, 1, 0, -1], [-3, -1, -2], [0, -5]):
    print(nums, "->", greedy(nums))

[1, 2, 3, 4, 5] -> 15
[1, 1, 0, 1, 1] -> 1
[1, 2, -1, -2, 1, 0, -1] -> 3
[-3, -1, -2] -> -1
[0, -5] -> 0


## 7. Các cách tiếp theo

Ta có 4 cách: (1) brute theo định nghĩa; (2) `set` + nhánh `max <= 0`; (3) sắp xếp giá trị khác nhau giảm dần và cộng khi còn dương; (4) mảng bucket 201 ô.

### Cách 2 — `set` các giá trị dương

Tính `m = max(nums)`; nếu `m <= 0` trả `m`; còn không trả `sum` của tập số dương. Một lần duyệt, gọn nhất.

In [9]:
class SolutionSet:
    def maxSum(self, nums):
        m = max(nums)
        if m <= 0:
            return m
        seen = set()
        total = 0
        for x in nums:
            if x > 0 and x not in seen:
                seen.add(x)
                total += x
        return total

print(SolutionSet().maxSum([1, 2, 3, 4, 5]))
print(SolutionSet().maxSum([1, 1, 0, 1, 1]))
print(SolutionSet().maxSum([1, 2, -1, -2, 1, 0, -1]))

15
1
3


Độ phức tạp Cách 2: Time `O(n)`, Space `O(n)` (tập `seen`).

### Cách 3 — Sắp xếp giảm dần các giá trị khác nhau

Lấy `vals = sorted(set(nums), reverse=True)`. Phần tử đầu là lớn nhất: nếu `<= 0` thì trả nó. Ngược lại cộng dần các phần tử cho tới khi gặp số không dương thì dừng (vì các số sau còn nhỏ hơn).

In [10]:
class SolutionSorted:
    def maxSum(self, nums):
        vals = sorted(set(nums), reverse=True)
        if vals[0] <= 0:
            return vals[0]
        total = 0
        for v in vals:
            if v <= 0:
                break
            total += v
        return total

print(SolutionSorted().maxSum([1, 2, 3, 4, 5]))
print(SolutionSorted().maxSum([1, 1, 0, 1, 1]))
print(SolutionSorted().maxSum([1, 2, -1, -2, 1, 0, -1]))

15
1
3


Độ phức tạp Cách 3: Time `O(n log n)` (sắp xếp), Space `O(n)`.

### Cách 4 — Mảng bucket 201 ô

Vì `-100 <= x <= 100`, dùng `seen[x + 100]`. Duyệt `v` từ `1` tới `100` cộng các giá trị dương đã thấy; nếu không có số dương thì trả phần tử lớn nhất đã thấy (duyệt từ `100` xuống `-100`, gặp ô đã thấy đầu tiên).

In [11]:
class SolutionBucket:
    def maxSum(self, nums):
        seen = [False] * 201
        for x in nums:
            seen[x + 100] = True
        total = 0
        has_positive = False
        for v in range(1, 101):
            if seen[v + 100]:
                total += v
                has_positive = True
        if has_positive:
            return total
        for v in range(100, -101, -1):          # không có số dương: lấy phần tử lớn nhất
            if seen[v + 100]:
                return v

print(SolutionBucket().maxSum([1, 2, 3, 4, 5]))
print(SolutionBucket().maxSum([1, 1, 0, 1, 1]))
print(SolutionBucket().maxSum([-3, -1, -2]))

15
1
-1


Độ phức tạp Cách 4: Time `O(n + V)` với `V = 201` hằng số, tức `O(n)`; Space `O(V) = O(1)`.

## 8. TRACE từng bước

Trace Cách 2 với `nums = [1, 2, -1, -2, 1, 0, -1]` (`max = 2 > 0`):

| bước | `x` | `x > 0`? | đã thấy? | `seen` | `total` |
|---|---|---|---|---|---|
| 1 | 1 | có | chưa | `{1}` | 1 |
| 2 | 2 | có | chưa | `{1,2}` | 3 |
| 3 | -1 | không | | `{1,2}` | 3 |
| 4 | -2 | không | | `{1,2}` | 3 |
| 5 | 1 | có | rồi | `{1,2}` | 3 |
| 6 | 0 | không | | `{1,2}` | 3 |
| 7 | -1 | không | | `{1,2}` | 3 |

Kết quả `3`.

In [12]:
def trace_unique(nums):
    m = max(nums)
    print("max =", m)
    if m <= 0:
        print("không có số dương -> trả max =", m)
        return m
    seen, total = set(), 0
    for step, x in enumerate(nums, start=1):
        if x > 0 and x not in seen:
            seen.add(x)
            total += x
            action = "cộng"
        elif x > 0:
            action = "trùng, bỏ qua"
        else:
            action = "không dương, bỏ qua"
        print(f"bước {step}: x={x:>2} {action:<22} seen={sorted(seen)} total={total}")
    return total

trace_unique([1, 2, -1, -2, 1, 0, -1])
trace_unique([-3, -1, -2])

max = 2
bước 1: x= 1 cộng                   seen=[1] total=1
bước 2: x= 2 cộng                   seen=[1, 2] total=3
bước 3: x=-1 không dương, bỏ qua    seen=[1, 2] total=3
bước 4: x=-2 không dương, bỏ qua    seen=[1, 2] total=3
bước 5: x= 1 trùng, bỏ qua          seen=[1, 2] total=3
bước 6: x= 0 không dương, bỏ qua    seen=[1, 2] total=3
bước 7: x=-1 không dương, bỏ qua    seen=[1, 2] total=3
max = -1
không có số dương -> trả max = -1


-1

## 9. Code nộp LeetCode

In [13]:
from typing import List

class Solution:
    def maxSum(self, nums: List[int]) -> int:
        m = max(nums)
        if m <= 0:
            return m
        return sum({x for x in nums if x > 0})

print(Solution().maxSum([1, 2, 3, 4, 5]), Solution().maxSum([1, 1, 0, 1, 1]), Solution().maxSum([1, 2, -1, -2, 1, 0, -1]))

15 1 3


## 10. Vì sao thuật toán đúng

- **Cận trên:** mảng con unique chỉ chứa các giá trị **khác nhau**, nên tổng không vượt quá tổng của một tập giá trị khác nhau. Mỗi giá trị âm hoặc `0` không giúp tổng lớn lên, vậy cận trên khi có số dương là **tổng các giá trị dương khác nhau**.
- **Đạt được cận trên:** giữ lại đúng một bản của mỗi giá trị dương khác nhau và xoá mọi phần tử khác. Mảng còn lại gồm các số dương đôi một khác nhau nên **cả mảng** là một mảng con unique với tổng đúng bằng cận trên. (Việc xoá cho phép chúng liền kề.) Mảng này khác rỗng vì có ít nhất một số dương.
- **Không có số dương:** mọi phần tử `<= 0`. Mảng con khác rỗng nào cũng có tổng `<=` phần tử lớn nhất trong nó `<= max(nums)` (mọi số cộng thêm không dương); và giữ riêng phần tử lớn nhất đạt đúng `max(nums)` (một phần tử luôn unique). Vậy đáp án là `max(nums)`.
- **Số `0`:** nếu có số dương, `0` không đổi tổng; nếu không có số dương và `0` có mặt thì `max = 0` -> đáp án `0`, đã được nhánh đầu xử lý.
- Cách 3, 4 chỉ là cách tính cùng tổng đó theo thứ tự/ bucket khác; Cách 1 chứng minh bằng vét cạn.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, giá trị nằm trong `[-100, 100]` nên `V = 201`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn mọi cách xoá | `O(2^n * n^2)` | `O(n)` |
| 2. `set` + nhánh `max <= 0` | `O(n)` | `O(n)` (`O(V)`) |
| 3. Sắp xếp giá trị khác nhau | `O(n log n)` | `O(n)` |
| 4. Bucket 201 ô | `O(n + V)` | `O(V) = O(1)` |

Cách 2 hoặc 4 là tối ưu. Cách 1 chỉ để kiểm chứng trên mảng nhỏ (`n <= 10`).

## 12. Bộ test

Test 1: ví dụ đề và các edge case (`n = 1`, toàn âm, toàn `0`, có `0` và âm, toàn trùng, biên `-100`/`100`).

In [14]:
impls = [SolutionBrute(), SolutionSet(), SolutionSorted(), SolutionBucket(), Solution()]
tests = [
    ([1, 2, 3, 4, 5], 15),
    ([1, 1, 0, 1, 1], 1),
    ([1, 2, -1, -2, 1, 0, -1], 3),
    ([5], 5),                          # n = 1, dương
    ([-7], -7),                        # n = 1, âm: bắt buộc giữ lại
    ([0], 0),
    ([-3, -1, -2], -1),                # toàn âm -> lấy lớn nhất, không phải 0
    ([0, -5, 0], 0),                   # max = 0
    ([3, 3, 3, 3], 3),                 # toàn trùng
    ([-100, 100], 100),                # biên
    ([-100, -100], -100),              # biên âm, trùng
    ([1, -5, 2], 3),                   # xoá -5 để 1 và 2 liền kề (bắt lỗi cửa sổ trượt không xoá)
    ([4, -1, 4, 2, -3, 2], 6),         # 4 + 2
]
for nums, exp in tests:
    assert 1 <= len(nums) <= 100 and all(-100 <= v <= 100 for v in nums)
    for im in impls:
        if isinstance(im, SolutionBrute) and len(nums) > 12:
            continue
        got = im.maxSum(list(nums))
        assert got == exp, (type(im).__name__, nums, got, exp)
print("tất cả test cố định OK:", len(tests))

tất cả test cố định OK: 13


Test 2: ngẫu nhiên có seed, đúng constraints. Mảng nhỏ (`n <= 9`) đối chiếu **cả brute force** (chân lý theo định nghĩa); mảng lớn (`n <= 100`) đối chiếu giữa các cách nhanh.

In [15]:
import random
random.seed(3487)

for t in range(600):
    n = random.randint(1, 9)
    lo = random.choice([-3, -100, 0])
    hi = random.choice([3, 100])
    nums = [random.randint(lo, hi) for _ in range(n)]
    assert 1 <= n <= 100 and all(-100 <= v <= 100 for v in nums)
    exp = SolutionBrute().maxSum(list(nums))
    for im in impls:
        assert im.maxSum(list(nums)) == exp, (type(im).__name__, nums)
print("600 test nhỏ (có brute) OK")

for t in range(800):
    n = random.randint(1, 100)
    nums = [random.randint(random.choice([-100, -5, 0]), random.choice([5, 100])) for _ in range(n)]
    assert all(-100 <= v <= 100 for v in nums)
    exp = SolutionSet().maxSum(list(nums))
    for im in impls[2:]:
        assert im.maxSum(list(nums)) == exp, (type(im).__name__, nums)
print("800 test lớn OK")

600 test nhỏ (có brute) OK
800 test lớn OK


Test 3: tính chất — không sửa input; đáp án không phụ thuộc thứ tự (xáo trộn); thêm một bản **sao** của phần tử bất kỳ không đổi đáp án; đáp án `>= max(nums)`; thêm số `-100` (nhỏ nhất) không đổi đáp án.

In [16]:
random.seed(87)
for _ in range(500):
    n = random.randint(1, 40)
    nums = [random.randint(-100, 100) for _ in range(n)]
    snap = list(nums)
    base = Solution().maxSum(nums)
    assert nums == snap
    sh = list(nums)
    random.shuffle(sh)
    assert Solution().maxSum(sh) == base
    assert Solution().maxSum(nums + [random.choice(nums)]) == base
    assert base >= max(nums)
    assert Solution().maxSum(nums + [-100]) == base
print("tính chất (không sửa input, xáo trộn, thêm bản sao, >= max, thêm -100): OK")

tính chất (không sửa input, xáo trộn, thêm bản sao, >= max, thêm -100): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: cộng cả số trùng (quên `unique`)

In [17]:
def wrong_duplicates(nums):
    m = max(nums)
    if m <= 0:
        return m
    return sum(x for x in nums if x > 0)      # SAI: cộng cả các bản trùng

nums = [1, 1, 0, 1, 1]
print("sai :", wrong_duplicates(nums))
print("đúng:", Solution().maxSum(nums))
assert wrong_duplicates(nums) != Solution().maxSum(nums)

sai : 4
đúng: 1


Vì sao sai: mảng con phải gồm các phần tử **đôi một khác nhau**, nên mỗi giá trị chỉ được dùng một lần: `[1,1,0,1,1]` chỉ cho `1`, không phải `4`. **Sửa:** dùng `set` (hoặc `seen`).

### Lỗi 2: trả `0` khi toàn số âm

In [18]:
def wrong_zero_floor(nums):
    return sum({x for x in nums if x > 0})    # SAI: toàn âm cho 0, nhưng mảng không được rỗng

nums = [-3, -1, -2]
print("sai :", wrong_zero_floor(nums))
print("đúng:", Solution().maxSum(nums))
assert wrong_zero_floor(nums) != Solution().maxSum(nums)

sai : 0
đúng: -1


Vì sao sai: tổng rỗng bằng `0` không được phép, vì đề bắt buộc mảng con **khác rỗng**. Khi mọi số `<= 0` phải giữ lại ít nhất một số, tốt nhất là số lớn nhất (`-1`). **Sửa:** `if max(nums) <= 0: return max(nums)`.

### Lỗi 3: cộng tổng các giá trị khác nhau **kể cả số âm**

In [19]:
def wrong_all_distinct(nums):
    return sum(set(nums))                     # SAI: số âm làm giảm tổng, nên xoá chúng

nums = [1, 2, -1, -2, 1, 0, -1]
print("sai :", wrong_all_distinct(nums))
print("đúng:", Solution().maxSum(nums))
assert wrong_all_distinct(nums) != Solution().maxSum(nums)

sai : 0
đúng: 3


Vì sao sai: `{1, 2, -1, -2, 0}` có tổng `0`, nhưng ta được **xoá** `-1` và `-2` để đạt `3`. **Sửa:** chỉ giữ giá trị dương.

### Lỗi 4: dùng cửa sổ trượt trên mảng gốc (quên rằng được xoá)

In [20]:
def wrong_sliding_window(nums):
    best = nums[0]
    for i in range(len(nums)):
        seen, total = set(), 0
        for j in range(i, len(nums)):
            if nums[j] in seen:
                break
            seen.add(nums[j])
            total += nums[j]
            best = max(best, total)
    return best                                 # SAI: không cho phép xoá phần tử ở giữa

nums = [1, -5, 2]
print("sai :", wrong_sliding_window(nums))
print("đúng:", Solution().maxSum(nums))
assert wrong_sliding_window(nums) != Solution().maxSum(nums)

sai : 2
đúng: 3


Vì sao sai: trên mảng gốc `[1, -5, 2]`, mảng con unique tốt nhất là `[2]` (tổng `2`). Nhưng đề cho phép **xoá** `-5` trước, còn `[1, 2]` (tổng `3`). Đây là bài "xoá tuỳ ý rồi chọn", không phải "tìm mảng con trên mảng gốc".

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề có "**xoá tuỳ ý rồi chọn mảng con**" -> thứ tự và khoảng cách **không còn quan trọng**, bài thành chọn **tập giá trị**.
- "Unique" + "tổng lớn nhất" -> gom **một bản** mỗi giá trị **dương** (tham lam); số `0` và âm không giúp.
- Luôn nghĩ đến nhánh **toàn không dương** vì mảng không được rỗng: đáp án `max(nums)`.
- Kiểm tra bằng **brute force trên mảng nhỏ** (mặt nạ bit) để chắc chắn trực giác đúng.
- Giá trị trong khoảng nhỏ `[-100, 100]` -> mảng bucket thay `set`.
- Đừng nhầm với bài `1695. Maximum Erasure Value` (cửa sổ trượt, **không** được xoá giữa chừng).

## 15. Biến thể của bài

1. **Maximum Erasure Value** (`1695`): chỉ được chọn mảng con unique **trên mảng gốc** (không xoá) -> cửa sổ trượt + `set`.
2. **Mỗi giá trị dùng tối đa `k` lần**: đáp án là tổng `min(count, k) * giá trị` của các số dương.
3. Yêu cầu trả về **độ dài** hoặc **chính mảng** đạt tổng lớn nhất.
4. Tổng phải **chia hết cho `m`** hoặc **bằng đúng `target`** (chuyển thành knapsack trên các giá trị khác nhau).
5. Được xoá **đúng một** phần tử (`1186. Maximum Subarray Sum with One Deletion`).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn mọi cách xoá | mặt nạ bit + mảng con unique | `O(2^n * n^2)` | `O(n)` | kiểm chứng trên mảng nhỏ |
| 2. `set` + nhánh `max <= 0` | cộng giá trị dương khác nhau | `O(n)` | `O(n)` | **chuẩn, nên nộp** |
| 3. Sắp xếp giá trị khác nhau | cộng từ lớn đến khi hết dương | `O(n log n)` | `O(n)` | dễ hình dung "lấy từ lớn xuống" |
| 4. Bucket 201 ô | `seen[x + 100]` | `O(n + V)` | `O(1)` | giá trị trong khoảng nhỏ |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `maxSum([2, 2, -1, 3, 3, -4])` bằng bao nhiêu?

**Bài 2 (tính tay):** `nums = [-5, -2, -2, -9]`: đáp án là gì và giữ phần tử nào?

**Bài 3 (điền TODO):** hoàn thành `distinct_positive_sum(nums)` trả về tổng các giá trị dương khác nhau (có thể `0` nếu không có số dương).

**Bài 4 (tự viết lại):** viết `best_selection(nums)` trả về **danh sách** các giá trị được giữ (đã sắp xếp tăng dần) để đạt đáp án (nếu không có số dương thì danh sách một phần tử `[max]`).

**Bài 5 (mini challenge):** viết `max_sum_each_k(nums, k)` — mỗi giá trị được dùng **tối đa `k` lần** trong mảng con (chấp nhận trùng tối đa `k` bản); trả về tổng lớn nhất, nhớ nhánh toàn không dương.

In [21]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().maxSum([2, 2, -1, 3, 3, -4]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def distinct_positive_sum(nums):
    # TODO: duyệt nums, cộng x nếu x > 0 và chưa thấy
    return 0

5


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** được xoá nên chỉ cần quan tâm các giá trị dương khác nhau; nếu không có số dương thì lấy phần tử lớn nhất.

**Mức 2 (vừa):** Bài 3: dùng `seen = set()`. Bài 4: `sorted({x for x in nums if x > 0})`, nếu rỗng trả `[max(nums)]`. Bài 5: với mỗi giá trị dương `v` có `c` bản, cộng `v * min(c, k)`; dùng `collections.Counter`.

**Mức 3 (gần đáp án):** Bài 1: giá trị dương khác nhau là `2` và `3` -> `5`. Bài 2: toàn âm, phần tử lớn nhất là `-2` -> đáp án `-2`, giữ `[-2]`.

## 19. Đáp án tham khảo cho bài tập

In [22]:
from collections import Counter

# Bài 1: các giá trị dương khác nhau 2, 3 -> 5
assert Solution().maxSum([2, 2, -1, 3, 3, -4]) == 5

# Bài 2: toàn âm -> max = -2
assert Solution().maxSum([-5, -2, -2, -9]) == -2

# Bài 3
def distinct_positive_sum(nums):
    seen = set()
    total = 0
    for x in nums:
        if x > 0 and x not in seen:
            seen.add(x)
            total += x
    return total
assert distinct_positive_sum([2, 2, -1, 3, 3, -4]) == 5
assert distinct_positive_sum([-1, 0]) == 0
assert distinct_positive_sum([7, 7, 7]) == 7

# Bài 4
def best_selection(nums):
    pos = sorted({x for x in nums if x > 0})
    return pos if pos else [max(nums)]
assert best_selection([1, 2, -1, -2, 1, 0, -1]) == [1, 2]
assert best_selection([-5, -2, -2, -9]) == [-2]
assert sum(best_selection([2, 2, -1, 3, 3, -4])) == Solution().maxSum([2, 2, -1, 3, 3, -4])

# Bài 5
def max_sum_each_k(nums, k):
    m = max(nums)
    if m <= 0:
        return m
    cnt = Counter(nums)
    return sum(v * min(c, k) for v, c in cnt.items() if v > 0)
assert max_sum_each_k([1, 1, 0, 1, 1], 1) == Solution().maxSum([1, 1, 0, 1, 1])
assert max_sum_each_k([1, 1, 0, 1, 1], 2) == 2
assert max_sum_each_k([-3, -1], 5) == -1
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "xoá tuỳ ý + chọn unique tổng lớn nhất":**

```python
m = max(nums)
if m <= 0:                         # không có số dương: buộc giữ ít nhất một số
    return m
return sum({x for x in nums if x > 0})   # mỗi giá trị dương khác nhau một lần
```

**Checklist trước khi submit:**

- [ ] Hiểu "xoá trước, chọn mảng con sau" -> không phải cửa sổ trượt trên mảng gốc?
- [ ] Mỗi giá trị dương chỉ cộng **một lần** (dùng `set`)?
- [ ] Không cộng số âm; số `0` không ảnh hưởng?
- [ ] Nhánh toàn không dương: trả `max(nums)` (không trả `0`)?
- [ ] Đã thử `n = 1`, toàn âm, toàn trùng, có `0`?
- [ ] Chữ ký đúng `maxSum(self, nums: List[int]) -> int`?